### Import Packages

In [14]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_cloudflare import ChatCloudflareWorkersAI
from typing import List
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from sentence_transformers import SentenceTransformer
import os
import re
import numpy as np
# from pydantic import BaseModel
# from langchain_openai import ChatOpenAI
# from langchain_anthropic import ChatAnthropic
# from langchain_core.output_parsers import PydanticOutputParser
# from tools import search_tool, wiki_tool, save_tool

load_dotenv()
cf_api_token = os.getenv("CF_AI_API_KEY")
cf_account_id = os.getenv("CF_ACCOUNT_ID")

### Just calling a model
Let's try calling a simple LLM first using CloudFlare Workers AI

In [2]:
llm = ChatCloudflareWorkersAI(
    api_token=cf_api_token,
    model="@cf/meta/llama-2-7b-chat-int8",
)
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content='Je adore le programmation.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 36, 'completion_tokens': 7, 'total_tokens': 43}, 'model_name': '@cf/meta/llama-2-7b-chat-int8'}, id='run--a79adb15-4147-4775-a79d-ea79f46f516a-0', usage_metadata={'input_tokens': 36, 'output_tokens': 7, 'total_tokens': 43})

In [3]:
print(ai_msg.content)

Je adore le programmation.


### Chaining model with a prompt separately

In [4]:
prompt = ChatPromptTemplate(
    [
        (
            "system",
            "You are a helpful assistant that translates {input_language} to {output_language}.",
        ),
        ("human", "{input}"),
    ]
)

chain = prompt | llm
chain.invoke(
    {
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }
)

AIMessage(content='Ich liebe Programmieren!', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 31, 'completion_tokens': 6, 'total_tokens': 37}, 'model_name': '@cf/meta/llama-2-7b-chat-int8'}, id='run--26094630-42b4-4a78-a552-b4e2b55ddac0-0', usage_metadata={'input_tokens': 31, 'output_tokens': 6, 'total_tokens': 37})

### Structured Outputs 

In [5]:
json_schema = {
    "title": "joke",
    "description": "Joke to tell user.",
    "type": "object",
    "properties": {
        "setup": {
            "type": "string",
            "description": "The setup of the joke",
        },
        "punchline": {
            "type": "string",
            "description": "The punchline to the joke",
        },
        "rating": {
            "type": "integer",
            "description": "How funny the joke is, from 1 to 10",
            "default": None,
        },
    },
    "required": ["setup", "punchline"],
}

llm = ChatCloudflareWorkersAI(
    api_token=cf_api_token,
    model="@cf/meta/llama-3.1-8b-instruct-fast",
)
structured_llm = llm.with_structured_output(json_schema)
structured_llm.invoke("Tell me a joke about cats")

{'punchline': 'Why did the cat join a band?',
 'rating': 7,
 'setup': 'Because it wanted to be the purr-cussionist!'}

### Bind Tools
We can create custom tools or use existing ones with the LLM to give it the ability to perform certain functions 

In [6]:
@tool
def validate_user(user_id: int, addresses: List[str]) -> bool:
    """Validate user using historical addresses.

    Args:
        user_id (int): the user ID.
        addresses (List[str]): Previous addresses as a list of strings.
    """
    return True


llm_with_tools = llm.bind_tools([validate_user])

result = llm_with_tools.invoke(
    "Could you validate user 123? They previously lived at "
    "123 Fake St in Boston MA and 234 Pretend Boulevard in "
    "Houston TX."
)
result.tool_calls

[{'name': 'validate_user',
  'args': {'user_id': 123,
   'addresses': '["123 Fake St Boston MA", "234 Pretend Boulevard Houston TX"]'},
  'id': 'b5a19f20-dd09-4b10-8e6f-c951320f7833',
  'type': 'tool_call'}]

### Custom Tools
Let's define tools for our document editing requirement

In [ ]:
# import faiss
# from langchain_community.vectorstores import FAISS
# import pickle

d:\__dev__\agentic_ai\test_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# Load embedding model from sentence transformers
embedder = SentenceTransformer('all-MiniLM-L6-v2')

In [15]:
DOCS_FOLDER = "./docs"
# INDEX_PATH = "docs_index.faiss"
# METADATA_PATH = "docs_metadata.pkl"

@tool
def search_text_file(query: str) -> str:
    """Search for a text file.

    Args:
        query (str): A description of what the file contains, e.g. 
                     "user onboarding", "error handling", "pricing details".

    Returns:
        str: A file path or file name that match the description.
    """
    # Gather all .txt files in docs folder
    file_paths = []
    texts = []
    for root, _, files in os.walk(DOCS_FOLDER):
        for f in files:
            if f.endswith(".txt"):
                path = os.path.join(root, f)
                file_paths.append(path)
                with open(path, "r", encoding="utf-8") as file:
                    texts.append(file.read())

    # Embed all docs
    doc_embeddings = embedder.encode(texts)
    print("all docs embedded", len(doc_embeddings))

    # Embed the query
    query_embedding = embedder.encode([query])[0]

    # Compute cosine similarity
    similarities = np.dot(doc_embeddings, query_embedding) / (
        np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding) + 1e-10
    )

    # Get the best matching doc
    best_idx = int(np.argmax(similarities))
    docs = file_paths[best_idx]
    return docs

@tool
def edit_text_file(file_path: str, find_text: str, replace_text: str) -> str:
    """Edit a text file by replacing text.

    Args:
        file_path (str): Path to the file that needs editing.
        find_text (str): The text to search for in the file.
        replace_text (str): The text to replace it with.

    Returns:
        str: A confirmation message with summary of changes.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    new_content = re.sub(find_text, replace_text, content)
    
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(new_content)

    return f"Replaced '{find_text}' with '{replace_text}' in {file_path}."

In [16]:
search_text_file("climate change")

C:\Users\karan\AppData\Local\Temp\ipykernel_31512\1427980197.py:1: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  search_text_file("climate change")


all docs embedded 6


'./docs\\climate.txt'

In [17]:
llm = ChatCloudflareWorkersAI(
    api_token=cf_api_token,
    model="@cf/mistralai/mistral-small-3.1-24b-instruct",
)

llm_with_tools = llm.bind_tools([search_text_file, edit_text_file])

result = llm_with_tools.invoke(
    "Find the text file that mentions climate change. Edit it by replacing the word 'climate' with 'weather'"
)
result.tool_calls
# by default the LLM will only pick one tool that seems relevant and stop there.
# it wont plan a multi-step sequence of actions.
# create_react_agent creates that loop!

[{'name': 'search_text_file',
  'args': {'query': 'climate change'},
  'id': '3923f991-014a-489f-b23b-4dac5f179af9',
  'type': 'tool_call'}]

In [18]:
agent = create_react_agent(llm, [search_text_file, edit_text_file])
query = input("Enter a query: ")
result = agent.invoke({"messages": [{"role": "user", "content": query}]})
result

all docs embedded 6


{'messages': [HumanMessage(content="i want to edit the file that talks about setting up windows, replace the word 'laptop' with 'Desktop' in that file", additional_kwargs={}, response_metadata={}, id='2c2380b2-f8d2-49c3-8778-1119a8ba9627'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 353, 'completion_tokens': 38, 'total_tokens': 391}, 'model_name': '@cf/mistralai/mistral-small-3.1-24b-instruct'}, id='run--ab7758ff-8cbb-4d76-b7fa-2e6712b9c05d-0', tool_calls=[{'name': 'search_text_file', 'args': {'query': 'setting up windows'}, 'id': 'c00580dc-2e98-407b-afe9-6515e54c2c6d', 'type': 'tool_call'}], usage_metadata={'input_tokens': 353, 'output_tokens': 38, 'total_tokens': 391}),
  ToolMessage(content='./docs\\windows2.txt', name='search_text_file', id='451c72d7-ae7e-4f34-8b93-39319547fd79', tool_call_id='c00580dc-2e98-407b-afe9-6515e54c2c6d'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens'

#### Langchains Available Tools
We can also call exisiting tools that are available under the langchain community

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults
search = DuckDuckGoSearchResults()
search.invoke("Best graphics cards in 2025?")

In [ ]:
llm = ChatCloudflareWorkersAI(
    api_token=cf_api_token,
    model="@cf/mistralai/mistral-small-3.1-24b-instruct",
)
agent = create_react_agent(llm, [search_text_file, edit_text_file, search])
query = input("Enter a query: ")
result = agent.invoke({"messages": [{"role": "user", "content": query}]})
result

In [ ]:
for msg in result["messages"]:
    print(msg, end="\n\n")